# Eternal Dao · Episode 01 · Private illustration batch

Generates approximately 200 unique 16:9 Chinese manhua illustrations for approved D03 Chapters 1–2. It privately reads the already approved Google Docs script and storyboard. **No images are displayed here**, no manual artwork approval is required, and no videos are compiled.

Select **Runtime → Change runtime type → T4 GPU** first, then **Runtime → Run all**. When asked, authorize the Google account that owns the approved source Docs and artwork folder. Already saved PNGs are reused on subsequent runs.

**Free Colab GPUs are limited and not guaranteed.** Initial model download takes additional time; the cells log real speed while generating images. Images are saved immediately to the existing private Drive folder. Colab interruptions are safe to resume.

In [ ]:
%pip -q install "diffusers==0.35.2" "transformers>=4.45,<5" "accelerate>=1.1" "safetensors" "huggingface_hub" "google-api-python-client"

In [ ]:
import hashlib, json, math, re, time
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable Runtime > Change runtime type > T4 GPU before running.")
print("GPU:", torch.cuda.get_device_name(0),
      "| VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
creds, _ = google.auth.default()
docs = build("docs", "v1", credentials=creds, cache_discovery=False)
SCRIPT_DOC_ID = "1FnoycxFSFwagkjz8RbWc1Ng9GTBtdkCAQ0aYtLd0MpI"
STORYBOARD_DOC_ID = "1vw0znZUigZs-q4ciu8-bGEFiJnRwTdEnltp0a63zF_E"
def google_doc_text(doc_id):
    doc = docs.documents().get(documentId=doc_id, includeTabsContent=True).execute()
    out = []
    def collect(contents):
        for node in contents or []:
            for el in node.get("paragraph", {}).get("elements", []):
                out.append(el.get("textRun", {}).get("content", ""))
            for row in node.get("table", {}).get("tableRows", []):
                for cell in row.get("tableCells", []):
                    collect(cell.get("content"))
    def visit(tab):
        collect(tab.get("documentTab", {}).get("body", {}).get("content", []))
        for child in tab.get("childTabs", []):
            visit(child)
    if doc.get("tabs"):
        for tab in doc["tabs"]:
            visit(tab)
    else:
        collect(doc.get("body", {}).get("content", []))
    return "".join(out)
SCRIPT = google_doc_text(SCRIPT_DOC_ID)
BOARD = google_doc_text(STORYBOARD_DOC_ID)
if "CHAPTER ONE" not in SCRIPT or "CHAPTER TWO" not in SCRIPT:
    raise RuntimeError("Wrong or inaccessible locked episode source")
print("Private source loaded:", len(SCRIPT.split()), "words |",
      len(BOARD), "storyboard characters (not publishing private text)")

In [ ]:
# The 24 storyboard beats are not 24 pictures. Split the verbatim episode
# into many ~18-word visual shots, without modifying any spoken narration.
WORDS = SCRIPT.split()
BEATS = [{"id": int(i), "title": t.strip(), "visual": v.strip()}
         for i, t, v in re.findall(r"(?m)^\s*(\d{2})\s+([^\n]+)\nVisual:\s*([^\n]+)", BOARD)]
if len(BEATS) != 24 or [b["id"] for b in BEATS] != list(range(1, 25)):
    raise RuntimeError("The original approved 24 storyboard beats could not be read")
cast_block = BOARD.split("LOCKED CHARACTER SHEETS", 1)[1].split("SCENE-BY-SCENE", 1)[0]
CAST = dict((name.strip(), info.strip()) for name, info in
            re.findall(r"(?m)^([A-Z][A-Z\s]+)\s+[—–-]\s*([^\n]+)", cast_block))
chapter_two = next((i for i in range(len(WORDS)-1)
                    if WORDS[i].upper().strip(".,:") == "CHAPTER"
                    and WORDS[i+1].upper().strip(".,:") == "TWO"), None)
if chapter_two is None:
    raise RuntimeError("Cannot locate Chapter Two boundary")
SHOTS_N = math.ceil(len(WORDS) / 18)
SHA = hashlib.sha256(SCRIPT.encode()).hexdigest()
TARGET = (Path("/content/drive/MyDrive/Audiobook_Studio") /
          "Eternal Dao of the Lost Epoch — D03 — Illustration Library" /
          "Episode 01 — Chapters 1–2 — Original Illustrations" /
          ("D03-" + SHA[:12]))
if not TARGET.parent.exists():
    raise RuntimeError("The existing private Google Drive illustration folder is unavailable")
IMAGES = TARGET / "illustrations"
IMAGES.mkdir(parents=True, exist_ok=True)
SHOTS = []
camera = ["wide establishing view", "medium character interaction", "expressive reaction close-up",
          "over-the-shoulder viewpoint", "detail of the relevant tool or object",
          "low architectural angle", "cinematic side profile", "tense three-quarter framing",
          "intimate dramatic close-up", "grounded physical action shot"]
style = ("Single full-frame 16:9 2D Chinese manhua illustration, original ink-defined "
         "drawn aesthetic, cinematic lighting, believable anatomy and practical technology "
         "blended with Chinese cultivation, dark brick city and copper industrial devices "
         "as appropriate to the narrated moment. NOT a storyboard collage. ")
exclude = ("No caption, no text, no watermarks, no extra panels, no dragons, no unexplained "
           "jade relics or mystical golden vortex, no floating fantasy pagodas, no unrelated "
           "sect battles or blood moon, no 3D render. ")
for index in range(SHOTS_N):
    start = index * len(WORDS) // SHOTS_N
    end = (index + 1) * len(WORDS) // SHOTS_N
    excerpt = " ".join(WORDS[start:end])
    nearby = " ".join(WORDS[max(0,start-20):min(len(WORDS),end+20)])
    if start < chapter_two:
        beat_index = min(11, int((start/max(1,chapter_two))*12))
    else:
        beat_index = 12 + min(11, int(((start-chapter_two)/max(1,len(WORDS)-chapter_two))*12))
    beat = BEATS[beat_index]
    context = (nearby + " " + beat["visual"]).lower()
    characters = []
    for name, description in CAST.items():
        aliases = [name.lower(), name.split()[-1].lower()]
        if any(len(a)>3 and re.search(r"\b"+re.escape(a)+r"\b", context) for a in aliases):
            characters.append(name + ": " + description)
    prompt = (style + "Narrative situation: " + beat["visual"] + ". "
              + ("Relevant cast: " + " / ".join(characters[:3])[:600] + ". " if characters else "")
              + "Exact nearby narration context: " + nearby[:460] + ". "
              + "Camera: " + camera[index % len(camera)] + ". "
              + "Depict this physical moment accurately, not generic fantasy artwork. " + exclude)
    source_hash = hashlib.sha256(excerpt.encode()).hexdigest()
    prompt_hash = hashlib.sha256((prompt+source_hash).encode()).hexdigest()
    SHOTS.append({"index":index, "start_word":start,"end_word":end,
                  "storyboard_beat":beat["id"],"source_excerpt":excerpt,
                  "source_sha256":source_hash,"prompt":prompt,
                  "prompt_sha256":prompt_hash,
                  "file":f"scene-{index:04d}-{prompt_hash[:12]}.png"})
if " ".join(s["source_excerpt"] for s in SHOTS) != " ".join(WORDS):
    raise RuntimeError("Narration coverage is not exact")
def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".part")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf8")
    temp.replace(path)
save_json(TARGET/"PRIVATE_EP01_SHOT_PLAN.json",
          {"source_sha256":SHA,"storyboard_sha256":hashlib.sha256(BOARD.encode()).hexdigest(),
           "owner_artwork_approval":"blanket_preapproved","total_shots":len(SHOTS),
           "characters":CAST,"shots":SHOTS})
print("Prepared",len(SHOTS),"unique illustrations from",len(WORDS),"exact narration words")
print("Private Drive directory:",TARGET)

In [ ]:
# SDXL-Lightning 4-step generation, automatically resumable from Drive.
# Change this to 2 only to trade visual quality for speed.
STEPS = 4
MAX_NEW_IMAGES_THIS_SESSION = 0  # 0 = attempt all remaining; free Colab can disconnect.
from PIL import Image
def valid_image(path):
    try:
        if not path.is_file() or path.stat().st_size < 12000:
            return False
        with Image.open(path) as image:
            size, fmt = image.size, image.format
            image.verify()
        return size == (1024,576) and fmt == "PNG"
    except (OSError, ValueError):
        return False
existing = sum(valid_image(IMAGES/s["file"]) for s in SHOTS)
print("Images already safely stored:", existing, "/", len(SHOTS))
if existing < len(SHOTS):
    from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
    from huggingface_hub import hf_hub_download
    print("Loading SDXL base + 4-step Lightning LoRA (first model download may take time)")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    ckpt = hf_hub_download("ByteDance/SDXL-Lightning",
                           f"sdxl_lightning_{STEPS}step_lora.safetensors")
    pipe.load_lora_weights(ckpt)
    pipe.fuse_lora()
    pipe.scheduler = EulerDiscreteScheduler.from_config(
        pipe.scheduler.config, timestep_spacing="trailing")
    pipe.to("cuda")
    pipe.enable_vae_slicing()
    pipe.set_progress_bar_config(disable=True)
    # Save one appearance reference per character. These are private assets.
    references = TARGET / "character_references"
    references.mkdir(parents=True, exist_ok=True)
    for j,(name,description) in enumerate(CAST.items()):
        dest = references/(re.sub(r"[^a-z0-9]+","-",name.lower()).strip("-")+".png")
        if valid_image(dest):
            continue
        portrait = (style+"Character appearance portrait reference: "+name+
                    ". "+description+". One figure, simple background, waist-up. "+exclude)
        with torch.inference_mode():
            image = pipe(portrait, num_inference_steps=STEPS, guidance_scale=0.0,
                         width=1024, height=576,
                         generator=torch.Generator("cuda").manual_seed(320000+j)).images[0]
        image.save(dest, format="PNG")
        print("Saved character reference:", name)
    new=0
    t0=time.monotonic()
    for shot in SHOTS:
        dest=IMAGES/shot["file"]
        if valid_image(dest):
            continue
        if MAX_NEW_IMAGES_THIS_SESSION and new >= MAX_NEW_IMAGES_THIS_SESSION:
            break
        with torch.inference_mode():
            img=pipe(shot["prompt"],num_inference_steps=STEPS,guidance_scale=0.0,
                     width=1024,height=576,
                     generator=torch.Generator("cuda").manual_seed(580000+shot["index"]*113)).images[0]
        temp=dest.with_suffix(".partial.png")
        img.save(temp,format="PNG")
        if not valid_image(temp):
            temp.unlink(missing_ok=True)
            raise RuntimeError("Generated invalid image at scene "+str(shot["index"]))
        temp.replace(dest)
        new+=1
        if new%10==0:
            print(f"New: {new} | Overall: {existing+new}/{len(SHOTS)} |"
                  f" recent average {((time.monotonic()-t0)/new):.1f} seconds/image")
            save_json(TARGET/"PRIVATE_EP01_PROGRESS.json",
                      {"saved_images":existing+new,"planned":len(SHOTS),
                       "owner_artwork_approval":"blanket_preapproved"})
    del pipe
    torch.cuda.empty_cache()
records=[]
for shot in SHOTS:
    file=IMAGES/shot["file"]
    if valid_image(file):
        records.append({"index":shot["index"],"storyboard_beat":shot["storyboard_beat"],
                        "file":"illustrations/"+shot["file"],
                        "source_sha256":shot["source_sha256"],
                        "image_sha256":hashlib.sha256(file.read_bytes()).hexdigest()})
manifest={"schema":"private-episode-illustration-pack-v1",
          "story_id":"eternal-dao-of-the-lost-epoch","episode":1,
          "script_source_sha256":SHA,"owner_artwork_approval":"blanket_preapproved",
          "planned_images":len(SHOTS),"verified_images":len(records),
          "complete":len(records)==len(SHOTS),
          "compilation_allowed":False,
          "note":"Full-novel illustration gate is separately required before video assembly",
          "images":records}
save_json(TARGET/"PRIVATE_EP01_ILLUSTRATION_MANIFEST.json",manifest)
print("Private Google Drive: verified",len(records),"of",len(SHOTS),"illustrations.")
print("Re-running this notebook skips correctly saved images; no pictures displayed.")
if manifest["complete"]:
    print("COMPLETE: Episode 1 illustration pack saved; video compilation not started.")